# 00. mindGAP demo (dummy data)

このノートブックは **実データを使わず**、合成した二値質問票データで
mindGAP 解析パイプラインの流れを確認するためのデモです。

流れ:
1. ダミー二値データ生成（被験者 × wave × item）
2. VEM で pairwise MEM（Ising 型）パラメータ \((h, J)\) を推定
3. 推定 \(J\) の要約・PCA
4. 少数被験者で exact fixation（\(2^9\) 列挙）を計算

前提: mindGAP コンテナ（`jaxenv`）または同等の JAX / scikit-learn 環境

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# notebooks/ から実行しても src / 同階層モジュールを拾えるようにする
NB_DIR = Path.cwd()
ROOT = NB_DIR if (NB_DIR / "VEM_MEM.py").exists() else NB_DIR.parent
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

OUT_DIR = Path("figs_for_paper/mindgap_demo")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("cwd:", Path.cwd())
print("OUT_DIR:", OUT_DIR.resolve())

## 1. ダミーデータ生成

ハブ項目（0-based 6,7,8）が周辺項目より ON しやすい構造を埋め込んだ合成データ。

In [ ]:
def make_dummy_binary_questionnaire(
    n_subjects: int = 80,
    n_waves: int = 5,
    n_items: int = 9,
    hub_items_0b: tuple[int, ...] = (6, 7, 8),
    seed: int = 42,
) -> np.ndarray:
    """Return array shape (N, T, d) with values in {0, 1}."""
    rng = np.random.default_rng(seed)

    # subject-level severity latent
    severity = rng.normal(0.0, 1.0, size=n_subjects)

    data = np.zeros((n_subjects, n_waves, n_items), dtype=np.int8)
    for s in range(n_subjects):
        for t in range(n_waves):
            wave_noise = rng.normal(0.0, 0.35)
            for i in range(n_items):
                base = -0.4 + 0.55 * severity[s] + wave_noise
                if i in hub_items_0b:
                    base += 0.9  # hubs more often ON
                # mild coupling to neighboring items via shared severity only
                p = 1.0 / (1.0 + np.exp(-base))
                data[s, t, i] = rng.random() < p
    return data


data = make_dummy_binary_questionnaire()
print("dummy data shape (N, T, d):", data.shape)
print("overall ON rate:", float(data.mean()))
print("per-item ON rate:", data.mean(axis=(0, 1)).round(3))

## 2. VEM fit → \((h, J)\)

In [ ]:
import jax.numpy as jnp
from VEM_MEM import VEMConfig, VEMMEM

config = VEMConfig(
    max_iter=2000,
    alpha_h_init=0.01,
    alpha_J_init=100,
    tol=1e-7,
)
model = VEMMEM(d=data.shape[-1], config=config)
results = model.fit(jnp.array(data), verbose=True, time_mode="aggregate")

h = np.asarray(results["mu_all"][:, 0, :9], dtype=float)
J = np.asarray(results["mu_all"][:, 0, 9:], dtype=float)
print("h shape:", h.shape)
print("J shape:", J.shape)
print("n_iterations:", results.get("n_iterations"))
print("final ELBO:", float(results.get("final_elbo", np.nan)))

## 3. \(J\) の被験者 PCA とハブ平均

In [ ]:
def edge_pairs(n_items: int = 9):
    return [(i, j) for i in range(n_items) for j in range(i + 1, n_items)]


pairs = edge_pairs(9)
hub = {6, 7, 8}
hub_edge_idx = [k for k, (i, j) in enumerate(pairs) if (i in hub) or (j in hub)]

mean_hub_J = J[:, hub_edge_idx].mean(axis=1)

pca = PCA(n_components=2, random_state=0)
coords = pca.fit_transform(J)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
sc = axes[0].scatter(coords[:, 0], coords[:, 1], c=mean_hub_J, cmap="coolwarm", s=35, edgecolors="k", linewidths=0.3)
fig.colorbar(sc, ax=axes[0], fraction=0.046, pad=0.04, label="mean hub-edge J")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[0].set_title("Subject PCA of J (dummy)")

axes[1].hist(mean_hub_J, bins=20, color="#4C72B0", edgecolor="k", alpha=0.85)
axes[1].set_xlabel("mean hub-edge J")
axes[1].set_ylabel("count")
axes[1].set_title("Hub-edge coupling distribution")
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.savefig(OUT_DIR / "demo_J_pca_and_hub_mean.pdf", bbox_inches="tight")
plt.show()
print("PCA explained:", pca.explained_variance_ratio_.round(3))

## 4. Exact fixation（少数被験者）

\(n=9\) なので \(2^9=512\) 状態を列挙して周辺確率から
\(\mathrm{fixation}=\max(p,1-p)\) を計算する。

In [ ]:
from scipy.special import logsumexp


def Jvec_to_Jmat(J_vec: np.ndarray, n_items: int = 9) -> np.ndarray:
    J_mat = np.zeros((n_items, n_items), dtype=float)
    idx = 0
    for i in range(n_items):
        for j in range(i + 1, n_items):
            J_mat[i, j] = J_vec[idx]
            J_mat[j, i] = J_vec[idx]
            idx += 1
    return J_mat


def exact_fixation(h_vec: np.ndarray, J_vec: np.ndarray) -> np.ndarray:
    n_items = len(h_vec)
    states = np.array([[(s >> i) & 1 for i in range(n_items)] for s in range(2 ** n_items)], dtype=np.int8)
    linear = states @ h_vec
    pair = np.zeros(states.shape[0], dtype=float)
    for k, (i, j) in enumerate(edge_pairs(n_items)):
        pair += J_vec[k] * states[:, i] * states[:, j]
    logw = linear + pair
    prob = np.exp(logw - logsumexp(logw))
    p_on = prob @ states
    return np.maximum(p_on, 1.0 - p_on)


n_show = min(12, h.shape[0])
fix_mat = np.vstack([exact_fixation(h[s], J[s]) for s in range(n_show)])

fig, ax = plt.subplots(figsize=(8, 3.8), constrained_layout=True)
sns.heatmap(fix_mat, ax=ax, cmap="viridis", vmin=0.5, vmax=1.0)
ax.set_xlabel("item (0-based)")
ax.set_ylabel("subject (first N)")
ax.set_title("Exact fixation on dummy VEM fits")
ax.axvline(6, color="white", lw=1.2)
fig.savefig(OUT_DIR / "demo_exact_fixation_heatmap.pdf", bbox_inches="tight")
plt.show()

print("mean fixation background(0-5):", fix_mat[:, :6].mean().round(3))
print("mean fixation target(6-8):", fix_mat[:, 6:].mean().round(3))

## 完了

ここまで動けば mindGAP の解析環境（JAX / VEM / 可視化）は一通り通っています。
実データ解析は別ノートブックで、`data/original/`（ローカルのみ・Git 管理外）を参照してください。